In [1]:
!pip install -q transformers torch

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "facebook/bart-large-mnli"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

print("Zero-shot classification model loaded successfully!")

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Zero-shot classification model loaded successfully!


In [3]:
def zero_shot_classify(text, candidate_labels):
    results = []

    for label in candidate_labels:
        hypothesis = f"This text is about {label}."

        inputs = tokenizer(
            text,
            hypothesis,
            return_tensors="pt",
            truncation=True,
            max_length=512
        )

        with torch.no_grad():
            outputs = model(**inputs)

        entailment_score = torch.softmax(
            outputs.logits, dim=1
        )[0, 2].item()

        results.append((label, entailment_score))

    results.sort(key=lambda x: x[1], reverse=True)

    return results

In [4]:
text = "My laptop keeps shutting down and I think the battery needs to be replaced."

candidate_labels = [
    "technology",
    "sports",
    "finance",
    "health"
]

results = zero_shot_classify(text, candidate_labels)

print("Text:")
print(text)

print("\nClassification Results:")

for label, score in results:
    print(f"{label:<15} {score:.2%}")

Text:
My laptop keeps shutting down and I think the battery needs to be replaced.

Classification Results:
technology      43.01%
health          5.55%
finance         2.10%
sports          0.37%


In [5]:
while True:
    text = input("\nEnter text (or type 'exit' to stop): ")

    if text.lower() == "exit":
        print("Zero-shot classifier stopped.")
        break

    labels_input = input(
        "Enter categories separated by commas: "
    )

    candidate_labels = [
        label.strip()
        for label in labels_input.split(",")
        if label.strip()
    ]

    results = zero_shot_classify(
        text,
        candidate_labels
    )

    print("\nClassification Results:")

    for label, score in results:
        print(f"{label:<20} {score:.2%}")

    print(f"\nBest Category: {results[0][0]}")


Enter text (or type 'exit' to stop): exit
Zero-shot classifier stopped.


In [6]:
def display_classification(text, candidate_labels):
    results = zero_shot_classify(text, candidate_labels)

    best_label, best_score = results[0]

    print("Text:")
    print(text)

    print("\nBest Category:")
    print(best_label)

    print(f"Confidence: {best_score:.2%}")

    print("\nAll Categories:")
    for label, score in results:
        print(f"- {label}: {score:.2%}")

In [7]:
text = "I want to learn Python and build artificial intelligence applications."

candidate_labels = [
    "technology",
    "sports",
    "finance",
    "education"
]

display_classification(text, candidate_labels)

Text:
I want to learn Python and build artificial intelligence applications.

Best Category:
technology
Confidence: 46.35%

All Categories:
- technology: 46.35%
- education: 0.83%
- finance: 0.30%
- sports: 0.09%
